# Bioassay: a complete Bayesian workflow in one small example

**Short Bayesian course — Example 1**

This notebook uses the classic bioassay data from Racine-Poon et al. (1986), as presented in Section 3.5 of Gelman & Vehtari's *Bayesian Workflow*.

The dataset is tiny on purpose: four doses, five animals at each dose. That makes it possible to see the entire Bayesian workflow without the computation or data structure getting in the way.

### Learning goals

By the end of the notebook, you should be able to:

1. write a grouped binomial logistic-regression model;
2. understand what the priors imply **before** looking at the fitted model;
3. fit the model with PyMC and inspect basic computational diagnostics;
4. interpret the posterior as uncertainty about an entire dose-response curve;
5. derive a scientifically meaningful quantity, the **LD50**, from posterior draws;
6. use posterior predictive simulation to ask whether the fitted model can reproduce data like those observed.

The central modeling question is simple:

> **How does the probability of death change with dose, and what dose corresponds to a 50% mortality probability?**


## 0. Setup

The notebook is written for current PyMC 6 / ArviZ 1.x.

If a Colab runtime has incompatible older packages, uncomment the installation line and run it once before the imports.


In [ ]:
# Uncomment in Colab only if the installed PyMC / ArviZ versions are incompatible.
# %pip install -q "pymc==6.3.2" "arviz==1.3.0"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

RANDOM_SEED = 20260921
rng = np.random.default_rng(RANDOM_SEED)

print("PyMC:", pm.__version__)
print("ArviZ:", az.__version__)

## 1. The data

At each of four dose levels, five animals were exposed and the number of deaths was recorded.

Because the outcome is a **count out of a known number of trials**, the natural observation model is binomial.

The dose values are on the log(g/ml) scale used in the original example.


In [ ]:
bioassay = pd.DataFrame(
    {
        "dose": [-0.86, -0.30, -0.05, 0.73],
        "n": [5, 5, 5, 5],
        "deaths": [0, 1, 3, 5],
    }
)

bioassay["death_rate"] = bioassay["deaths"] / bioassay["n"]
bioassay

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(bioassay["dose"], bioassay["death_rate"], s=70)
ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Observed proportion dead",
    ylim=(-0.05, 1.05),
    title="Bioassay data",
)

plt.show()

## 2. A generative model

For dose group \(j\),

\[
y_j \sim \operatorname{Binomial}(n_j, p_j)
\]

and we model the death probability with logistic regression:

\[
\operatorname{logit}(p_j) = \alpha + \beta x_j.
\]

Equivalently,

\[
p_j = \operatorname{logistic}(\alpha + \beta x_j).
\]

We use

\[
\alpha \sim \mathcal N(0,5),
\qquad
\beta \sim \operatorname{HalfNormal}(5).
\]

The positive constraint on \(\beta\) encodes substantive knowledge that mortality should not *decrease* as dose increases. This is the PyMC equivalent of the positive-slope prior used in the *Bayesian Workflow* case study.

### Interpretation of the parameters

- \(\alpha\) is the log-odds of death when dose \(x=0\).
- \(\beta\) controls how rapidly the log-odds change with dose.
- Neither parameter is the main scientific target. The quantity we ultimately care about is a **function of them**: the LD50.

Before fitting the model, we should ask what these priors imply for possible dose-response curves.


In [ ]:
coords = {"dose_group": np.arange(len(bioassay))}

with pm.Model(coords=coords) as model:
    dose = pm.Data(
        "dose",
        bioassay["dose"].to_numpy(),
        dims="dose_group",
    )
    n = pm.Data(
        "n",
        bioassay["n"].to_numpy(),
        dims="dose_group",
    )

    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.HalfNormal("beta", sigma=5)

    logit_p = alpha + beta * dose
    p = pm.Deterministic(
        "p",
        pm.math.sigmoid(logit_p),
        dims="dose_group",
    )

    deaths = pm.Binomial(
        "deaths",
        n=n,
        p=p,
        observed=bioassay["deaths"].to_numpy(),
        dims="dose_group",
    )

model

## 3. Prior predictive thinking

A prior on regression coefficients is easier to understand after translating it into the **observable scale**.

Here we draw values of \(\alpha\) and \(\beta\) from the prior and turn each draw into a complete dose-response curve.

Do not ask only whether \(N(0,5)\) "sounds broad." Ask:

> **What kinds of mortality curves does that prior say are plausible?**


In [ ]:
with model:
    prior = pm.sample_prior_predictive(
        samples=1000,
        random_seed=RANDOM_SEED,
    )

alpha_prior = np.asarray(prior["prior"]["alpha"]).reshape(-1)
beta_prior = np.asarray(prior["prior"]["beta"]).reshape(-1)

x_grid = np.linspace(-1.0, 1.0, 200)

draw_ids = rng.choice(len(alpha_prior), size=60, replace=False)
prior_curves = 1 / (
    1 + np.exp(
        -(
            alpha_prior[draw_ids, None]
            + beta_prior[draw_ids, None] * x_grid[None, :]
        )
    )
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(x_grid, prior_curves.T, alpha=0.12, linewidth=1)
ax.scatter(
    bioassay["dose"],
    bioassay["death_rate"],
    s=70,
    zorder=5,
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Probability of death",
    ylim=(-0.05, 1.05),
    title="Curves implied by the prior",
)

plt.show()

### Pause before fitting

The prior permits a wide range of monotone increasing curves, including very steep transitions.

That is useful to notice now. Once we condition on the data, it becomes easy to forget that the posterior is partly determined by assumptions we made *before* seeing those data.

For this first example, we will keep the book's broad prior rather than tune it further.


## 4. Condition on the data

Bayes' rule conceptually gives us

\[
p(\alpha,\beta \mid y)
\propto
p(y\mid \alpha,\beta)\,
p(\alpha,\beta).
\]

PyMC lets us specify the model directly and uses MCMC to generate draws from the posterior.

Each posterior draw is one plausible pair \((\alpha,\beta)\), and therefore one plausible dose-response curve.


In [ ]:
with model:
    idata = pm.sample(
        draws=1000,
        tune=1000,
        chains=4,
        target_accept=0.9,
        random_seed=RANDOM_SEED,
    )

In [ ]:
az.summary(
    idata,
    var_names=["alpha", "beta"],
    hdi_prob=0.90,
    round_to=2,
)

The summary is partly about the scientific posterior and partly about the computation.

For the computational side, check that:

- \(\hat R\) is very close to 1;
- effective sample sizes are comfortably large;
- the sampler reports no divergences.

For this tiny model those checks should be uneventful. Later examples will give us more interesting diagnostics.


In [ ]:
az.plot_trace(idata, var_names=["alpha", "beta"])
plt.show()

## 5. The posterior is a distribution over curves

The coefficients are not usually the easiest way to think about a logistic-regression posterior.

Every posterior draw of \((\alpha,\beta)\) implies a complete function

\[
p(x) = \operatorname{logistic}(\alpha+\beta x).
\]

So uncertainty about parameters becomes uncertainty about the **dose-response relationship itself**.


In [ ]:
alpha_post = np.asarray(idata["posterior"]["alpha"]).reshape(-1)
beta_post = np.asarray(idata["posterior"]["beta"]).reshape(-1)

posterior_curves = 1 / (
    1 + np.exp(
        -(
            alpha_post[:, None]
            + beta_post[:, None] * x_grid[None, :]
        )
    )
)

curve_median = np.median(posterior_curves, axis=0)
curve_lo, curve_hi = np.quantile(posterior_curves, [0.05, 0.95], axis=0)

fig, ax = plt.subplots(figsize=(7, 4))

ax.fill_between(
    x_grid,
    curve_lo,
    curve_hi,
    alpha=0.25,
    label="Central 90% posterior interval",
)
ax.plot(
    x_grid,
    curve_median,
    linewidth=2,
    label="Posterior median",
)
ax.scatter(
    bioassay["dose"],
    bioassay["death_rate"],
    s=70,
    zorder=5,
    label="Observed proportions",
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Probability of death",
    ylim=(-0.05, 1.05),
    title="Posterior dose-response curve",
)
ax.legend()

plt.show()

## 6. A derived scientific quantity: LD50

The **LD50** is the dose at which the model predicts a 50% probability of death.

At \(p=0.5\), the log-odds are zero:

\[
0 = \alpha + \beta\,LD50.
\]

Therefore,

\[
LD50 = -\frac{\alpha}{\beta}.
\]

This is an important Bayesian move:

> We do **not** need to fit a second model for LD50. We transform every posterior draw of the original parameters.

The posterior distribution of a scientifically meaningful derived quantity follows automatically.


In [ ]:
ld50_log_g_ml = -alpha_post / beta_post
ld50_mg_ml = 1000 * np.exp(ld50_log_g_ml)

q05, q50, q95 = np.quantile(ld50_mg_ml, [0.05, 0.50, 0.95])

print(f"Posterior median LD50: {q50:.0f} mg/ml")
print(f"Central 90% credible interval: {q05:.0f} to {q95:.0f} mg/ml")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.hist(ld50_mg_ml, bins=50, density=True, alpha=0.75)
ax.axvline(q50, linestyle="--", label="Posterior median")

ax.set(
    xlabel="LD50 (mg/ml)",
    ylabel="Posterior density",
    title="Posterior distribution of LD50",
)
ax.legend()

plt.show()

### Interpretation

The LD50 is uncertain because \(\alpha\) and \(\beta\) are uncertain.

Notice the distinction between:

- **parameter uncertainty:** we do not know the exact dose-response curve;
- **outcome uncertainty:** even if the death probability at a dose were known exactly, a group of five animals would not always produce the same number of deaths.

Posterior predictive simulation includes **both** sources of uncertainty.


## 7. Posterior predictive checking

Now we ask the model to generate new bioassay outcomes at the same four doses.

For every posterior draw:

1. take its implied death probability \(p_j\);
2. simulate a new count

\[
\tilde y_j \sim \operatorname{Binomial}(5,p_j).
\]

These replicated datasets are things the fitted model says we could plausibly have observed.

A posterior predictive check compares the real data with those replications.


In [ ]:
with model:
    ppc = pm.sample_posterior_predictive(
        idata,
        var_names=["deaths"],
        random_seed=RANDOM_SEED,
        return_inferencedata=False,
    )

y_rep = np.asarray(ppc["deaths"]).reshape(-1, len(bioassay))

pred_lo, pred_med, pred_hi = np.quantile(
    y_rep,
    [0.05, 0.50, 0.95],
    axis=0,
)

fig, ax = plt.subplots(figsize=(7, 4))

ax.errorbar(
    bioassay["dose"],
    pred_med,
    yerr=[pred_med - pred_lo, pred_hi - pred_med],
    fmt="o",
    capsize=5,
    label="Posterior predictive median and central 90%",
)

ax.scatter(
    bioassay["dose"],
    bioassay["deaths"],
    s=80,
    marker="x",
    label="Observed deaths",
)

ax.set(
    xlabel="Dose, log(g/ml)",
    ylabel="Deaths out of 5",
    ylim=(-0.4, 5.4),
    title="Can the fitted model reproduce data like ours?",
)
ax.legend()

plt.show()

### What is this check doing?

This is **not** another estimate of the parameters.

We have already conditioned on the data. Now we use the fitted model to simulate observations and ask whether its predictions resemble the data in ways we care about.

With only four groups, the check cannot be very elaborate. That is a feature for this first example: the logic is visible.

Later, with richer data, posterior predictive checks can target patterns such as:

- skewness or heavy tails;
- nonlinear relationships;
- heteroskedasticity;
- subgroup differences;
- temporal or trial-to-trial structure.

That is where posterior predictive checking becomes much more powerful than simply inspecting fitted coefficients.


## 8. What this example gave us

In one small model we have used the complete workflow:

\[
\boxed{
\text{data}
\rightarrow
\text{generative model}
\rightarrow
\text{prior implications}
\rightarrow
\text{posterior}
\rightarrow
\text{derived quantity}
\rightarrow
\text{replicated data}
}
\]

Three ideas are worth carrying to the next examples:

1. **A Bayesian model is generative.**  
   It specifies how unknown quantities and observable data relate probabilistically.

2. **The posterior is more than a table of parameter estimates.**  
   Posterior draws can be transformed into curves, predictions, contrasts, thresholds, or any other scientifically meaningful quantity.

3. **Prediction is part of checking the model, not only forecasting the future.**  
   Posterior predictive simulation asks what the model can generate and whether those generated data resemble the observations.


## 9. Short exercises

These are intended for a few minutes of exploration rather than a full assignment.

### Exercise 1 — What does the monotonicity assumption buy us?

Replace

```python
beta = pm.HalfNormal("beta", sigma=5)
```

with

```python
beta = pm.Normal("beta", mu=0, sigma=5)
```

Fit the model again.

- How much posterior probability is placed on \(\beta<0\)?
- Does the LD50 posterior change appreciably?
- Was the positive-slope constraint doing substantial work, or merely encoding something the data already strongly suggested?

### Exercise 2 — Change the prior

Try narrower priors, for example

\[
\alpha \sim N(0,2), \qquad \beta \sim HalfNormal(2).
\]

Look at the prior curves **before** fitting.

- Which curves disappeared?
- Does the posterior change much?
- Does the LD50 change more or less than the regression coefficients?

### Exercise 3 — Predict at a new dose

Choose a new dose, for example \(x=0.2\).

For every posterior draw calculate

\[
p_{\text{new}}
=
\operatorname{logistic}(\alpha+\beta\,0.2).
\]

Then simulate the number of deaths in a new group of five animals.

What is the difference between the posterior distribution of \(p_{\text{new}}\) and the posterior predictive distribution of the **number of deaths**?

### Exercise 4 — Find a better posterior predictive check

The plot above checks each dose separately.

Can you define a single feature of the complete four-point dataset that would reveal a failure of the assumed monotone logistic shape? Simulate that feature from the posterior predictive distribution and compare it with the observed value.


## Sources

- Gelman, A. & Vehtari, A. *Bayesian Workflow*, Section 3.5, “A simple example of probabilistic programming.”  
  https://avehtari.github.io/Bayesian-Workflow/bioassay/bioassay.html
- Racine-Poon, A., Grieve, A. P., Fluhler, H., & Smith, A. F. M. (1986). “Bayesian Methods in Practice: Experiences in the Pharmaceutical Industry.” *Applied Statistics*, 35, 93–150.
- PyMC documentation: https://www.pymc.io/

This notebook follows the statistical model used in the *Bayesian Workflow* case study, translated into current PyMC and reorganized for teaching.
